# 03 — A/B Testing

Synthetic experimentation notebook covering experiment analysis and common pitfalls.

**Project:** Marketing Analytics Causal & LTV Lab  
**Style:** Hands-on, advanced, interview-ready notebook  
**How to use:** Run cell by cell, inspect outputs, then discuss interpretation and pitfalls.


## Main notebook code

Run this notebook and then we will discuss the output, assumptions and pitfalls.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
RANDOM_STATE=42; rng=np.random.default_rng(RANDOM_STATE)
SYNTHETIC_DIR=Path('../data/synthetic'); REPORTS_DIR=Path('../reports')
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True); REPORTS_DIR.mkdir(parents=True, exist_ok=True)
n=50000
ab=pd.DataFrame({'user_id':np.arange(n),'pre_period_spend':rng.gamma(2,20,n),'engagement_score':rng.normal(0,1,n)})
ab['treatment']=rng.binomial(1,0.5,n)
base=-2.4+0.015*ab['pre_period_spend']+0.25*ab['engagement_score']
ab['conversion_prob']=1/(1+np.exp(-(base+0.15*ab['treatment'])))
ab['converted']=rng.binomial(1,ab['conversion_prob'])
ab['revenue']=ab['converted']*rng.gamma(2,30,n)
def proportion_test(data):
    s=data.groupby('treatment')['converted'].agg(['sum','count','mean']); c0,n0,p0=s.loc[0,'sum'],s.loc[0,'count'],s.loc[0,'mean']; c1,n1,p1=s.loc[1,'sum'],s.loc[1,'count'],s.loc[1,'mean']; pooled=(c0+c1)/(n0+n1); se=np.sqrt(pooled*(1-pooled)*(1/n0+1/n1)); z=(p1-p0)/se; p=2*(1-stats.norm.cdf(abs(z))); return {'control_rate':p0,'treatment_rate':p1,'absolute_lift':p1-p0,'relative_lift':(p1-p0)/p0,'p_value':p}
display(pd.DataFrame([proportion_test(ab)]))
# SRM check
obs=ab['treatment'].value_counts().sort_index().values; exp=np.array([len(ab)*0.5,len(ab)*0.5]); print('SRM p-value:', stats.chisquare(obs, exp).pvalue)
# Peeking
looks=[]
ordered=ab.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
for end in range(2000,len(ordered)+1,2000):
    r=proportion_test(ordered.iloc[:end]); r['n_seen']=end; looks.append(r)
looks=pd.DataFrame(looks); display(looks.head())
plt.figure(figsize=(8,4)); plt.plot(looks['n_seen'],looks['p_value'],marker='o'); plt.axhline(0.05,linestyle='--'); plt.title('Peeking p-values'); plt.show()
# CUPED
theta=np.cov(ab['revenue'],ab['pre_period_spend'])[0,1]/np.var(ab['pre_period_spend'])
ab['revenue_cuped']=ab['revenue']-theta*(ab['pre_period_spend']-ab['pre_period_spend'].mean())
print('Raw t-test:', stats.ttest_ind(ab.loc[ab.treatment==1,'revenue'],ab.loc[ab.treatment==0,'revenue'],equal_var=False))
print('CUPED t-test:', stats.ttest_ind(ab.loc[ab.treatment==1,'revenue_cuped'],ab.loc[ab.treatment==0,'revenue_cuped'],equal_var=False))
ab.to_csv(SYNTHETIC_DIR/'synthetic_ab_testing.csv', index=False)


## Discussion prompts

1. What assumption is strongest here?
2. Which pitfall would break the conclusion?
3. How would you explain this to a non-technical stakeholder?
